In [45]:
import numpy as np
from collections import defaultdict

import tifffile
import numpy as np
import os

def evaluate_segmentation_3d(
    gt_labels: np.ndarray,
    pred_labels: np.ndarray,
    iou_threshold: float = 0.5,
    ior_threshold: float = 0.5
) -> dict:
    """
    Evaluates 3D instance segmentation performance, calculating TP, FP, FN,
    and specifically identifying merge and split errors.

    The logic follows these steps:
    1.  Find all overlapping pairs of ground truth (GT) and predicted labels.
    2.  Calculate IoU (Intersection over Union) and IoR (Intersection over Reference,
        i.e., over GT area) for each pair.
    3.  Map predictions to GT labels if IoR > ior_threshold. This allows a
        single prediction to match multiple GT labels (a potential merge).
    4.  Identify and classify all matches (one-to-one, merge, split) and errors.
    5.  Compile a detailed DataFrame of each event.
    6.  Return both a summary dictionary and the detailed DataFrame.

    Args:
        gt_labels (np.ndarray): A 3D labeled image for the ground truth.
        pred_labels (np.ndarray): A 3D labeled image for the prediction.
        iou_threshold (float): IoU threshold to be considered a True Positive.
        ior_threshold (float): IoR threshold to establish a potential match
                               between a prediction and a ground truth object.

    Returns:
        dict: A dictionary containing:
              'summary': A dict with the overall metrics ('tp', 'fp', 'fn', etc.).
              'details': A pandas DataFrame with a detailed breakdown of each object.
    """
    # Get unique labels from GT and prediction, ignoring background (label 0)
    gt_label_ids = np.unique(gt_labels[gt_labels > 0])
    pred_label_ids = np.unique(pred_labels[pred_labels > 0])

    # --- Step 1: Pre-calculate all pairwise metrics for efficiency ---
    pairwise_metrics = []
    gt_masks = {label_id: (gt_labels == label_id) for label_id in gt_label_ids}
    pred_masks = {label_id: (pred_labels == label_id) for label_id in pred_label_ids}

    for p_id in pred_label_ids:
        pred_mask = pred_masks[p_id]
        overlapping_gt_ids = np.unique(gt_labels[pred_mask])
        for gt_id in overlapping_gt_ids:
            if gt_id == 0: continue
            gt_mask = gt_masks[gt_id]
            intersection = np.sum(pred_mask & gt_mask)
            union = np.sum(pred_mask | gt_mask)
            gt_area = np.sum(gt_mask)
            iou = intersection / union if union > 0 else 0
            ior = intersection / gt_area if gt_area > 0 else 0
            pairwise_metrics.append({'p_id': p_id, 'gt_id': gt_id, 'iou': iou, 'ior': ior})

    # --- Step 2: Establish Mappings based on IoR threshold ---
    pred_to_gt_map = defaultdict(list)
    gt_to_pred_map = defaultdict(list)
    for metrics in pairwise_metrics:
        if metrics['ior'] > ior_threshold:
            pred_to_gt_map[metrics['p_id']].append(metrics)
    for p_id, gt_list in pred_to_gt_map.items():
        for gt_info in gt_list:
            gt_to_pred_map[gt_info['gt_id']].append(gt_info)

    # --- Step 3: Classify matches and build detailed results list ---
    tp, fp, fn = 0, 0, 0
    merge_errors, split_errors = 0, 0
    processed_preds, processed_gts = set(), set()
    results_list = []

    # Identify and process MERGE errors (1 pred -> N GTs)
    for p_id, gt_list in pred_to_gt_map.items():
        if len(gt_list) > 1:
            merge_errors += 1
            processed_preds.add(p_id)
            gt_list.sort(key=lambda x: x['iou'], reverse=True)
            best_gt_match = gt_list[0]
            
            # Log the primary interaction for the merge error
            results_list.append({
                'pred_label': p_id, 'gt_label': best_gt_match['gt_id'],
                'IoR': best_gt_match['ior'], 'IoU': best_gt_match['iou'], 'type': 'merge_error'
            })
            
            if best_gt_match['iou'] > iou_threshold:
                tp += 1
            else:
                fp += 1
                fn += 1
            processed_gts.add(best_gt_match['gt_id'])
            
            for other_gt in gt_list[1:]:
                fn += 1
                processed_gts.add(other_gt['gt_id'])
                # Log the other GTs involved in the merge as FNs
                results_list.append({
                    'pred_label': None, 'gt_label': other_gt['gt_id'],
                    'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
                })

    # Identify and process SPLIT errors (M preds -> 1 GT)
    for gt_id, p_list in gt_to_pred_map.items():
        if gt_id in processed_gts or len(p_list) <= 1: continue
        split_errors += 1
        processed_gts.add(gt_id)
        p_list.sort(key=lambda x: x['iou'], reverse=True)
        best_p_match = p_list[0]
        
        results_list.append({
            'pred_label': best_p_match['p_id'], 'gt_label': gt_id,
            'IoR': best_p_match['ior'], 'IoU': best_p_match['iou'], 'type': 'split_error'
        })
        
        if best_p_match['iou'] > iou_threshold:
            tp += 1
        else:
            fp += 1; fn += 1
        processed_preds.add(best_p_match['p_id'])
        
        for other_p in p_list[1:]:
            fp += 1
            processed_preds.add(other_p['p_id'])
            results_list.append({
                'pred_label': other_p['p_id'], 'gt_label': None,
                'IoR': np.nan, 'IoU': np.nan, 'type': 'fp'
            })

    # Classify clean one-to-one matches
    for p_id, gt_list in pred_to_gt_map.items():
        if p_id in processed_preds: continue
        gt_id = gt_list[0]['gt_id']
        if gt_id in processed_gts: continue
        
        iou = gt_list[0]['iou']
        ior = gt_list[0]['ior']
        
        if iou > iou_threshold:
            tp += 1
            results_list.append({
                'pred_label': p_id, 'gt_label': gt_id, 'IoR': ior, 'IoU': iou, 'type': 'tp'
            })
        else:
            fp += 1; fn += 1
            # A low IoU match counts as both an FP and an FN. We log them as two events in the table.
            results_list.append({
                'pred_label': p_id, 'gt_label': gt_id, 'IoR': ior, 'IoU': iou, 'type': 'fp'
            })
            results_list.append({
                'pred_label': None, 'gt_label': gt_id, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
            })
        processed_preds.add(p_id)
        processed_gts.add(gt_id)

    # --- Step 4: Log remaining unmatched objects ---
    unmatched_preds = set(pred_label_ids) - processed_preds
    fp += len(unmatched_preds)
    for p_id in unmatched_preds:
        results_list.append({
            'pred_label': p_id, 'gt_label': None, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fp'
        })

    unmatched_gts = set(gt_label_ids) - processed_gts
    fn += len(unmatched_gts)
    for gt_id in unmatched_gts:
        results_list.append({
            'pred_label': None, 'gt_label': gt_id, 'IoR': np.nan, 'IoU': np.nan, 'type': 'fn'
        })

    # --- Step 5: Final Assembly ---
    summary_dict = {'tp': tp, 'fp': fp, 'fn': fn, 'merge_errors': merge_errors, 'split_errors': split_errors}
    details_df = pd.DataFrame(results_list, columns=['pred_label', 'gt_label', 'IoR', 'IoU', 'type'])
    # Convert nullable integer columns
    details_df['pred_label'] = details_df['pred_label'].astype('Int64')
    details_df['gt_label'] = details_df['gt_label'].astype('Int64')
    
    return {'summary': summary_dict, 'details': details_df}





In [7]:
import os
import pandas as pd
gt_path = 'D:\\Mikala\\images\\GT\\corrected_GT'
pred_path = 'D:\\Mikala\\images\\tunning\\cp4_3d\\VASA_ch1'

pred_fileList = [f for f in os.listdir(pred_path)  if f.endswith(('masks.tif','masks.tiff'))]  
gt_fileList = [os.path.splitext(f)[0]+'_corrected.tif' for f in pred_fileList ] 


qc_details = []
qc_summary = []

for idx,f in enumerate(gt_fileList):
    print('\n[processFiles] processing file ' + str(idx+1) + ' of ' +str(len(gt_fileList)))
    print('[processFiles] Filename = ' + f)

    gt_labels =np.array(tifffile.imread(os.path.join(gt_path,f)))
    pred_labels =np.array(tifffile.imread(os.path.join(pred_path,pred_fileList[idx])))
    
    ior_threshold = 0.5
    iou_threshold = 0.5

    evaluation = evaluate_segmentation_3d(gt_labels, pred_labels, iou_threshold, ior_threshold)
    summary = pd.DataFrame(evaluation['summary'], index = [0] ).reset_index(drop=True)
    details_df = evaluation['details']

    PPV = summary['tp']/(summary['tp'] + summary['fp']) #Precision
    Sensitivity = summary['tp'][0]/(summary['tp'][0] + summary['fn'][0]) #recall
    F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)
    merge_errors = summary['merge_errors'][0]/len(pd.unique(details_df['gt_label']))
    split_errors = summary['split_errors'][0]/len(pd.unique(details_df['gt_label']))

    summary.insert(loc=0, column = 'total_cells_gt', value = len(pd.unique(details_df['gt_label'])))
    summary.insert(loc=0, column = 'split_error_rate', value = split_errors )
    summary.insert(loc=0, column = 'merge_error_rate', value = merge_errors )

    summary.insert(loc=0, column = 'PPV', value = PPV )
    summary.insert(loc=0, column = 'Sensitivity', value = Sensitivity)
    summary.insert(loc=0, column = 'F1', value = F1)

    summary.insert(loc=0, column = 'file', value = f)

    
    

  
    details_df.insert(loc=0, column = 'file', value = f)

    qc_details.append(details_df)
    qc_summary.append(summary)

qc_details = pd.concat(qc_details, ignore_index=True)
qc_summary = pd.concat(qc_summary, ignore_index=True)
qc_total_summary = qc_summary.sum(axis=0) 

PPV = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fp']) #Precision
Sensitivity = qc_total_summary['tp']/(qc_total_summary['tp'] + qc_total_summary['fn']) #recall

F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)

print ('Precision: ' + str(PPV))
print ('Sensitivity: '+ str(Sensitivity))
print ('F1 Score: ', str(F1))
print('Merge  error rate: ',  qc_total_summary['merge_errors']/qc_total_summary['total_cells_gt'])
print('Split error rate: ',  qc_total_summary['split_errors']/qc_total_summary['total_cells_gt'])
        


[processFiles] processing file 1 of 3
[processFiles] Filename = 02222025_control_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 2 of 3
[processFiles] Filename = 02222025_cora_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 3 of 3
[processFiles] Filename = 02262025_atp_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif
Precision: 0.8917647058823529
Sensitivity: 0.9381188118811881
F1 Score:  0.9143546441495777
Merge  error rate:  0.02457002457002457
Split error rate:  0.0


In [47]:
import os
import pandas as pd
gt_path = 'D:\\Mikala\\images\\GT\\corrected_GT'
pred_path = ['D:\\Mikala\\images\\tunning\\cp_current',
             'D:\\Mikala\\images\\tunning\\cp_current_3D',
             'D:\\Mikala\\images\\tunning\\cp_core',
             'D:\\Mikala\\images\\tunning\\cp_core_3D',
             'D:\\Mikala\\images\\tunning\\cp4_stitch01',
             'D:\\Mikala\\images\\tunning\\cp4_3D',
             'D:\\Mikala\\images\\tunning\\cp4_3D_300',
             'D:\\Mikala\\images\\tunning\\cp4_3D_300_smooth2',
             'D:\\Mikala\\images\\tunning\\cp4_3D_500']
pred_subfolder = 'VASA_ch1'

gt_fileList = [f for f in os.listdir(gt_path)  if f.endswith(('corrected.tif','corrected.tiff'))]
pred_fileList = [f.split('_corrected')[0] + '.tif' for f in gt_fileList ] 




qc_details = []
qc_summary = []
qc_model_summary = []

for i, pred in enumerate (pred_path):
    print('\n processing model ' + str(i+1) + ' of ' +str(len(pred_path)))
    #gt_fileList = gt_fileList[0:2]
    for idx,f in enumerate(gt_fileList):
        print('\n[processFiles] processing file ' + str(idx+1) + ' of ' +str(len(gt_fileList)))
        print('[processFiles] Filename = ' + f)

        gt_labels =np.array(tifffile.imread(os.path.join(gt_path,f)))
        pred_labels =np.array(tifffile.imread(os.path.join(pred,pred_subfolder,pred_fileList[idx])))
        
        ior_threshold = 0.5
        iou_threshold = 0.5

        evaluation = evaluate_segmentation_3d(gt_labels, pred_labels, iou_threshold, ior_threshold)
        summary = pd.DataFrame(evaluation['summary'], index = [0] ).reset_index(drop=True)
        details_df = evaluation['details']

        # Calculate per dataset metrics      
        PPV = summary['tp']/(summary['tp'] + summary['fp']) #Precision
        Sensitivity = summary['tp'][0]/(summary['tp'][0] + summary['fn'][0]) #recall
        F1 = (2*PPV*Sensitivity) /(PPV + Sensitivity)
        merge_errors = summary['merge_errors'][0]/len(pd.unique(details_df['gt_label']))
        split_errors = summary['split_errors'][0]/len(pd.unique(details_df['gt_label']))

        # Insert per dataset metrics
        summary.insert(loc=0, column = 'total_cells_gt', value = len(pd.unique(details_df['gt_label'])))  
        summary.insert(loc=0, column = 'split_error_rate', value = split_errors )
        summary.insert(loc=0, column = 'merge_error_rate', value = merge_errors )      
        summary.insert(loc=0, column = 'PPV', value = PPV )
        summary.insert(loc=0, column = 'Sensitivity', value = Sensitivity)
        summary.insert(loc=0, column = 'F1', value = F1)

        summary.insert(loc=0, column = 'file', value = f)
        summary.insert(loc=0, column = 'model', value = os.path.basename(pred))

        details_df.insert(loc=0, column = 'file', value = f)
        details_df.insert(loc=0, column = 'model', value = os.path.basename(pred))
       

        qc_details.append(details_df)
        qc_summary.append(summary)

    # Calculate per model metrics
    model_summary = summary.sum(axis=0, numeric_only = True).to_frame().T 

    model_summary['merge_error_rate'] = model_summary['merge_errors']/model_summary['total_cells_gt']
    model_summary['split_error_rate']  = model_summary['split_errors']/model_summary['total_cells_gt']
    model_summary['PPV'] = model_summary['tp']/(model_summary['tp'] + model_summary['fp']) #Precision
    model_summary['Sensitivity'] = model_summary['tp']/(model_summary['tp'] + model_summary['fn']) #recall
    model_summary['F1'] = (2*PPV*Sensitivity) /(PPV + Sensitivity)
    model_summary.insert(loc=0, column = 'model', value = os.path.basename(pred))
    qc_model_summary.append(model_summary)

    #qc_details = pd.concat(qc_details, ignore_index=True)
    #qc_summary = pd.concat(qc_summary, ignore_index=True)

    #Print model results
    print('\n',os.path.basename(pred),' Model performance:' )
    print ('\tPrecision(PPV): ' + str(model_summary.loc[0,'PPV']))
    print ('\tSensitivity: '+ str(model_summary.loc[0,'Sensitivity']))
    print ('\tF1 Score: ', str(model_summary.loc[0,'F1']))
    print('\tMerge  error rate: ', str(model_summary.loc[0,'merge_error_rate']))
    print('\tSplit error rate: ',  str(model_summary.loc[0,'split_error_rate']))

qc_details = pd.concat(qc_details, ignore_index=True)
qc_summary = pd.concat(qc_summary, ignore_index=True)
qc_model_summary = pd.concat(qc_model_summary, ignore_index=True)




 processing model 1 of 9

[processFiles] processing file 1 of 3
[processFiles] Filename = 02222025_control_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 2 of 3
[processFiles] Filename = 02222025_cora_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 3 of 3
[processFiles] Filename = 02262025_atp_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

 cp_current  Model performance:
	Precision(PPV): 0.8198198198198198
	Sensitivity: 0.6946564885496184
	F1 Score:  0.7520661157024794
	Merge  error rate:  0.1893939393939394
	Split error rate:  0.0

 processing model 2 of 9

[processFiles] processing file 1 of 3
[processFiles] Filename = 02222025_control_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 2 of 3
[processFiles] Filename = 02222025_cora_dapi_488_555_647_ovary1_ch1_downs_cp_masks_corrected.tif

[processFiles] processing file 3 of 3
[processFiles] Filename = 0

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Mikala\\images\\tunning\\cp4_3D_500\\VASA_ch1\\02222025_control_dapi_488_555_647_ovary1_ch1_downs_cp_masks.tif'

In [48]:

qc_details = pd.concat(qc_details, ignore_index=True)
qc_summary = pd.concat(qc_summary, ignore_index=True)
qc_model_summary = pd.concat(qc_model_summary, ignore_index=True)

In [50]:

import seaborn as sns
sns.set_theme(style="whitegrid")

# Draw a nested barplot by species and sex
qc_model_summary.plot
g = sns.barplot(
    data=qc_model_summary, kind="bar",
    x="model", y=["F1",'PPV'],
     palette="dark", alpha=.6, height=6
)
g.despine(left=True)

g.legend.set_title("")

ValueError: Length of list vectors must match length of `data` when both are used, but `data` has length 8 and the vector passed to `y` has length 2.